# PropCare AI — Stage 2: Multi-Agent Operations with LangGraph

A single support agent is limited when one request needs multiple domain checks, a durable workflow thread, and a decision that an AI must not make. Stage 2 makes those controls explicit with a LangGraph supervisor, three specialists, shared state, a checkpointer, and a raw human-in-the-loop interrupt.

## Architecture

The real `StateGraph` uses the `supervisor`, `maintenance`, `billing`, and `resident_services` nodes. It follows a genuine cycle: **Supervisor → Maintenance / Billing / Resident Services → Supervisor**. The second supervisor pass decides whether another specialist is needed, whether a financial approval must pause the workflow, or whether the result can finish. This hand-built orchestration is intentionally visible in Stage 2.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'backend').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

TENANT_ID = 'T-1001'
BILLING_MESSAGE = 'Can you check whether my rent for this month has been paid?'
COMPENSATION_MESSAGE = 'My heater has failed again and I want compensation.'

In [ ]:
import inspect
import backend.stage2.propcare_graph as stage2
from backend.stage2.propcare_graph import CHECKPOINTER, PropCareState, STAGE2_GRAPH

print('Explicit state fields:')
for field, annotation in PropCareState.__annotations__.items():
    print(f'- {field}: {annotation}')
print('\nCheckpointer:', type(CHECKPOINTER).__name__)
print('\nProduction StateGraph builder (nodes, conditional edges, and the specialist → supervisor cycle):')
print(inspect.getsource(stage2.build_stage2_graph))
print('\nCompiled graph (Mermaid):')
print(STAGE2_GRAPH.get_graph().draw_mermaid())

## Real, read-only cycle

The billing-only example below invokes the existing compiled Stage 2 graph. It follows Supervisor → Billing → Supervisor → Final and uses a fresh `thread_id` in the in-memory checkpointer. It is safe to run because a billing-only request reads payment data and never creates a maintenance request.

In [ ]:
import uuid
from backend.stage2.propcare_graph import get_stage2_state, start_stage2

thread_id, result = start_stage2(TENANT_ID, BILLING_MESSAGE, thread_id=f'stage2-notebook-{uuid.uuid4()}')
state_snapshot = get_stage2_state(thread_id)
print({
    'thread_id': thread_id,
    'status': result.get('status'),
    'specialists_used': result.get('completed_specialists'),
    'resolution': result.get('resolution'),
    'checkpoint_status': state_snapshot.get('status'),
})

## Human-in-the-loop for compensation

For a recurring heater problem with a compensation request, Maintenance first gathers service evidence; the Supervisor then routes to Billing; the Supervisor detects a proposed credit and sends the workflow to the approval node. Maintenance status and financial approval remain separate. The AI recommends a credit; a property manager chooses the outcome.

In [ ]:
import inspect
from langgraph.types import Command
import backend.stage2.propcare_graph as stage2

# This displays the actual production node that calls raw interrupt().
print(inspect.getsource(stage2.approval))

# An admin action resumes the paused graph with this production payload shape.
resume_payload = {'decision': 'approve', 'approved_credit': 5000, 'note': 'Approved by property manager'}
resume_command = Command(resume=resume_payload)
print('\nCommand(resume=...) prepared:', resume_command)

## Optional compensation run

The following real graph call can create or reuse a demo work order and place an approval in the local demo queue. It is opt-in so this explanatory notebook does not change the shared data by default. In the application, the admin console resumes the same thread with `Command(resume=...)`.

In [ ]:
RUN_COMPENSATION_DEMO = False
if RUN_COMPENSATION_DEMO:
    approval_thread, paused = start_stage2(TENANT_ID, COMPENSATION_MESSAGE)
    print({'thread_id': approval_thread, 'interrupted': bool(paused.get('__interrupt__')), 'state': paused})
else:
    print('Compensation run skipped. Set RUN_COMPENSATION_DEMO = True to create/reuse demo workflow data intentionally.')

## Stage 2 takeaway

LangGraph gives the application explicit state fields, nodes, edges, conditional routing, checkpoints, and pause/resume control. That precision is valuable, but it also means the supervisor and orchestration code must be maintained directly. Stage 3 explores a higher-level Deep Agents coordinator while preserving the same safety boundary.